# The far-right's success in European elections
This notebook analyses the far-right's success in European elections. The two main sources of data is ParlGov for election results and PopuList for the classification of far-right parties.

Load in the necessary libraries.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)

This analysis is based on the parlgov dataset. This dataset contains election results from all over the world for decades but also includes a right_left scale. 

However, this dataset only runs till mid-2023, so we had to update it with more recent election results. More about this later.

In [2]:
df = pd.read_csv('data/parl_gov/view_election.csv')

df['election_date'] = pd.to_datetime(df['election_date'])

Only keep rows where election type is parliamentary

In [3]:
df = df[df['election_type'] == 'parliament']

Filter to only include EU countries, the UK, Norway and Switzerland.

In [4]:
eu_countries = [
    "Austria",
    "Belgium",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "Czech Republic",
    "Denmark",
    "Estonia",
    "Finland",
    "France",
    "Germany",
    "Greece",
    "Hungary",
    "Ireland",
    "Italy",
    "Latvia",
    "Lithuania",
    "Luxembourg",
    "Malta",
    "Netherlands",
    "Poland",
    "Portugal",
    "Romania",
    "Slovakia",
    "Slovenia",
    "Spain",
    "Sweden",
    "United Kingdom",
    "Norway",
    "Switzerland"
]

df = df[df['country_name'].isin(eu_countries)]
df

,country_name_short,country_name,election_type,election_date,vote_share,seats,seats_total,party_name_short,party_name,party_name_english,left_right,country_id,election_id,previous_parliament_election_id,previous_cabinet_id,party_id
295,AUT,Austria,parliament,1919-02-16,40.75,72.0,170,SPO,Sozialdemokratische Partei Österreichs,Social Democratic Party of Austria,3.7293,59,1030,NaN,NaN,973
296,AUT,Austria,parliament,1919-02-16,35.93,69.0,170,OVP,Österreichische Volkspartei,Austrian People's Party,6.4733,59,1030,NaN,NaN,1013
297,AUT,Austria,parliament,1919-02-16,5.85,8.0,170,DNP,Deutschnationale,German-Nationals,7.4000,59,1030,NaN,NaN,2672
298,AUT,Austria,parliament,1919-02-16,1.90,5.0,170,GFOP,Deutsche Freiheits und Ordnungspartei,German Freedom and Order Party,8.7000,59,1030,NaN,NaN,2676
299,AUT,Austria,parliament,1919-02-16,1.49,4.0,170,one-seat,one-seat,one-seat,NaN,59,1030,NaN,NaN,2686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8927,SWE,Sweden,parliament,2022-09-11,6.75,24.0,349,V,Vänsterpartiet (kommunisterna),Left Party (Communists),1.5464,35,1112,1041.0,1652.0,882
8928,SWE,Sweden,parliament,2022-09-11,5.34,19.0,349,KD,Kristdemokraterna,Christian Democrats,7.1950,35,1112,1041.0,1652.0,282
8929,SWE,Sweden,parliament,2022-09-11,5.08,18.0,349,MP,Miljöpartiet de Gröna,Greens,3.3789,35,1112,1041.0,1652.0,1154
8930,SWE,Sweden,parliament,2022-09-11,4.61,16.0,349,FP,Folkpartiet,People's Party,6.2906,35,1112,1041.0,1652.0,892


Another way to define far-right parties is by using PopuList's definition of far-right parties. Their data can be joined on the ParlGov data using the `parlgov_id`.

In [5]:
far_right = pd.read_csv('data/The PopuList 4.0.csv', sep=';')

far_right = far_right[far_right['parlgov_id'].notna()] # drop rows where parl_gov ID is missing
far_right['parlgov_id'] = far_right['parlgov_id'].round().astype(int) # round the parl_gov ID to the nearest integer

Limit the number of columns we use from the PopuList dataset.

In [6]:
far_right = far_right[['party_name', 'party_name_short', 'farright', 'farright_start', 'farright_end', 'parlgov_id']]

Join far_right with df on `parlgov_id`.

In [7]:
merged_df = df.merge(far_right, left_on='party_id', right_on='parlgov_id', how='left')
merged_df

,country_name_short,country_name,election_type,election_date,vote_share,seats,seats_total,party_name_short_x,party_name_x,party_name_english,left_right,country_id,election_id,previous_parliament_election_id,previous_cabinet_id,party_id,party_name_y,party_name_short_y,farright,farright_start,farright_end,parlgov_id
0,AUT,Austria,parliament,1919-02-16,40.75,72.0,170,SPO,Sozialdemokratische Partei Österreichs,Social Democratic Party of Austria,3.7293,59,1030,NaN,NaN,973,NaN,NaN,NaN,NaN,NaN,NaN
1,AUT,Austria,parliament,1919-02-16,35.93,69.0,170,OVP,Österreichische Volkspartei,Austrian People's Party,6.4733,59,1030,NaN,NaN,1013,NaN,NaN,NaN,NaN,NaN,NaN
2,AUT,Austria,parliament,1919-02-16,5.85,8.0,170,DNP,Deutschnationale,German-Nationals,7.4000,59,1030,NaN,NaN,2672,NaN,NaN,NaN,NaN,NaN,NaN
3,AUT,Austria,parliament,1919-02-16,1.90,5.0,170,GFOP,Deutsche Freiheits und Ordnungspartei,German Freedom and Order Party,8.7000,59,1030,NaN,NaN,2676,NaN,NaN,NaN,NaN,NaN,NaN
4,AUT,Austria,parliament,1919-02-16,1.49,4.0,170,one-seat,one-seat,one-seat,NaN,59,1030,NaN,NaN,2686,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5778,SWE,Sweden,parliament,2022-09-11,6.75,24.0,349,V,Vänsterpartiet (kommunisterna),Left Party (Communists),1.5464,35,1112,1041.0,1652.0,882,Vänsterpartiet,V,0.0,2100.0,2100.0,882.0
5779,SWE,Sweden,parliament,2022-09-11,5.34,19.0,349,KD,Kristdemokraterna,Christian Democrats,7.1950,35,1112,1041.0,1652.0,282,NaN,NaN,NaN,NaN,NaN,NaN
5780,SWE,Sweden,parliament,2022-09-11,5.08,18.0,349,MP,Miljöpartiet de Gröna,Greens,3.3789,35,1112,1041.0,1652.0,1154,NaN,NaN,NaN,NaN,NaN,NaN
5781,SWE,Sweden,parliament,2022-09-11,4.61,16.0,349,FP,Folkpartiet,People's Party,6.2906,35,1112,1041.0,1652.0,892,NaN,NaN,NaN,NaN,NaN,NaN


Then we copy the dataframe and move it to google sheets, so we can update the data with the most recent election results.

In [8]:
merged_df.to_clipboard()

The complete data can be found [here](https://docs.google.com/spreadsheets/d/15bi1iuvPbuZcib2Nd-aOapouTSkE9Ax-1JdPMeBsaJs/edit?gid=1306356820#gid=1306356820). Now we load it in.

In [9]:
df = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vRHLJhncF4_LRS1jkmRGdgREfL1c6rmxTFMrygbrfEadzxWGp12HQI_TbO5O7fhz0fL5Mj3uq0RRp0q/pub?gid=1306356820&single=true&output=csv')

Remove duplicate rows on date, country_name, party and vote_share

In [10]:
df = df.drop_duplicates(subset=['election_date', 'country_name', 'party_name_english', 'vote_share'])

For the parties that don't have farright or left_right or party_id look if there is a previous row with the same party name and fill in the missing values accordingly.

In [11]:
df[['farright', 'left_right', 'party_id','farright_start','farright_end','parlgov_id']] = df.groupby(['country_name','party_name_english'])[['farright', 'left_right', 'party_id','farright_start','farright_end','parlgov_id']].ffill()

# also forwardfill on parlgov id 
df[['farright', 'left_right', 'farright_start','farright_end','parlgov_id']] = df.groupby(['party_id'])[['farright', 'left_right', 'farright_start','farright_end','parlgov_id']].ffill()

Create a new far_right column which is 'far-right' if `farright == 1` and otherwise "non far-right".

In [12]:
df['far_right'] = ((df['farright'] == 1)).map({True: 'far-right', False: 'non far-right'})

After we added the newest election results, we ended up with some new parties, that can fairly be categorized as far-right. Some of them (like United Right which includes the Polish far-right party Law and Justice) is a new form of an old far-right party while others (like Party for Young People) are entirely new.

In [13]:
other_far_right = [{"party_name_english": "Fidesz -- Hungarian Civic Party / Christian Democratic People's Party", "country_name": "Hungary"},
                    {"party_name_english": "United Right", "country_name": "Poland"},
                    {"party_name_english": "The Citizens' Party", "country_name": "Denmark"},
                    {"party_name_english": "Greatness", "country_name": "Bulgaria"},
                    {"party_name_english": "S.O.S Romania", "country_name": "Romania"},
                    {"party_name_english": "Party of Young People", "country_name": "Romania"},
                    {"party_name_english": "Motorists", "country_name": "Czech Republic"},
                    {"party_name_english": "People and Justice Union", "country_name": "Lithuania"},
                    {"party_name_english": "Republic Movement", "country_name": "Slovakia"}]

Update df so that parties listed in other_far_right are marked as far_right if they match on party name and country.

In [14]:
for party in other_far_right:
    df.loc[(df['party_name_english'] == party['party_name_english']) & (df['country_name'] == party['country_name']), 'far_right'] = "far-right"

In [15]:
# Find the first election date for each country where there is a vote share
first_election_dates = df[df['vote_share'].notna()].groupby('country_name')['election_date'].min()

# drop election dates before that for each country
df = df.merge(first_election_dates.rename('first_election_date'), on='country_name')
df = df[df['election_date'] >= df['first_election_date']]
df = df.drop(columns='first_election_date')

In [16]:
df

,country_name_short,country_name,election_type,election_date,vote_share,seats,seats_total,party_name_short_x,party_name_x,party_name_english,left_right,country_id,election_id,previous_parliament_election_id,previous_cabinet_id,party_id,party_name_y,party_name_short_y,farright,farright_start,farright_end,parlgov_id,far_right
7,HUN,Hungary,parliament,1990-04-08,11.7,44.0,386.0,FKgP,Független Kisgazda Párt,Independent Small Holders Party,9.0186,39.0,257.0,NaN,NaN,870.0,Független Kisgazdapárt,FKgP,1.0,2019.0,2100.0,870.0,far-right
8,GRC,Greece,parliament,1990-04-08,46.9,150.0,300.0,ND,Néa Đimokratía,New Democracy,6.7365,41.0,284.0,508.0,716.0,47.0,NaN,NaN,NaN,NaN,NaN,NaN,non far-right
9,GRC,Greece,parliament,1990-04-08,38.6,123.0,300.0,PASOK,Panellinio Sosialistikó Kínima,Panhellenic Socialist Movement,4.4968,41.0,284.0,508.0,716.0,1338.0,NaN,NaN,NaN,NaN,NaN,NaN,non far-right
10,GRC,Greece,parliament,1990-04-08,10.3,19.0,300.0,SYN,Synaspismós tīs Aristerás,Coalition of the Left,2.7683,41.0,284.0,508.0,716.0,1441.0,Synaspismós tis Aristerás,SYN,0.0,2100.0,2100.0,1441.0,non far-right
11,GRC,Greece,parliament,1990-04-08,NaN,4.0,300.0,none,no party affiliation,no party affiliation,NaN,41.0,284.0,508.0,716.0,1358.0,NaN,NaN,NaN,NaN,NaN,NaN,non far-right
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3205,SVN,Slovenia,parliament,2026-03-22,NaN,1.0,90.0,Ita,Italijanska narodna skupnost,Italian national community,NaN,60.0,1108.0,1025.0,1668.0,199.0,NaN,NaN,NaN,NaN,NaN,NaN,non far-right
3206,SVN,Slovenia,parliament,2026-03-22,5.4,5.0,90.0,L,Levica,The Left,1.3000,60.0,1108.0,1025.0,1668.0,2670.0,Levica,Levica,0.0,2100.0,2100.0,2670.0,non far-right
3207,SVN,Slovenia,parliament,2026-03-22,6.7,6.0,90.0,ZL-SD,Združena lista – Socialni demokrati,United List -- Social Democrats,3.0637,60.0,1108.0,1025.0,1668.0,706.0,Združena levica,ZL,0.0,2100.0,2100.0,706.0,non far-right
3208,SVN,Slovenia,parliament,2026-03-22,9.3,9.0,90.0,NSI,Nova Slovenija – Krščanska ljudska stranka,New Slovenia -- Christian People's Party,7.9345,60.0,1108.0,1025.0,1668.0,1047.0,Nova Slovenija – Krščanski Demokrati,N.Si,1.0,1900.0,2100.0,1047.0,far-right


The data only includes parties that either got at least one seat in parliament or at least 1 percent of the votes in each election. This means that vote_share doesn't sum to 100. Therefore, we create a other category that fills the remaining vote share to reach 100%

In [17]:
for (country_name, election_date), group in df.groupby(["country_name", "election_date"]):
    total_vote_share = group["vote_share"].sum()
    if total_vote_share < 100:
        other_row = {
            "country_name": country_name,
            "election_date": election_date,
            "party_name_english": "Other",
            "vote_share": 100 - total_vote_share,
            "far_right": "other"
        }
        df = pd.concat([df, pd.DataFrame([other_row])], ignore_index=True)

Limit the results to after 1994.

In [18]:
# change election date to datetime
df['election_date'] = pd.to_datetime(df['election_date'])

# filter it go 1994 or later
df = df[df['election_date'].dt.year >= 1994]

# create a year column
df['election_year'] = df['election_date'].dt.year

PopuList also register if a party is either becoming far-right or ceasing to be far-right. We only want to categorize parties as far-right if they are currently far-right. Therefore we update the far-right column based on the `farright_start`and `farright_end` columns.

In [19]:
num_far_right_before = df[df['far_right'] == 'far-right'].shape[0]

df.loc[df['farright_start'] > df['election_year'], 'far_right'] = 'non far-right'
df.loc[df['farright_end'] < df['election_year'], 'far_right'] = 'non far-right'

num_far_right_after = df[df['far_right'] == 'far-right'].shape[0]
print(f"This changed the number of far-right parties across the elections from {num_far_right_before} to {num_far_right_after}")

This changed the number of far-right parties across the elections from 390 to 357


In [20]:
df.to_csv("data/full_data.csv", index=False, sep=";")

Calculate the share of votes to far-right parties in the most recent election per country

In [21]:
far_right_share = df[(df['far_right'] == 'far-right')].groupby('country_name').apply(lambda x: x[x['election_date'] == x['election_date'].max()]['vote_share'].sum()).reset_index(name='far_right_share')
far_right_share = far_right_share.sort_values(by='far_right_share', ascending=False)
far_right_share.tail()

/var/folders/hx/m3n3wwr91yg_snnv68ynqsp80000gn/T/ipykernel_28351/1278159904.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  far_right_share = df[(df['far_right'] == 'far-right')].groupby('country_name').apply(lambda x: x[x['election_date'] == x['election_date'].max()]['vote_share'].sum()).reset_index(name='far_right_share')


,country_name,far_right_share
2,Bulgaria,10.38
14,Latvia,9.29
16,Luxembourg,9.27
11,Greece,4.44
15,Lithuania,1.38
